## 0 · The Challenge

> **Six weeks after Palermo International signed:** Riverside House has deployed its editorial
> assistant. Junior editors have embraced it for author background notes, manuscript summaries,
> and acquisition briefs. Then a senior editor notices something in a brief sent to an author:
>
> _"Elena Marchetti won the Booker Prize in 2019 for her debut novel **The Silence of Bridges**."_
>
> Elena Marchetti was **longlisted** for the Booker in 2019. She did not win.

The system's metrics at the time of that answer:

| Metric                   | Score   | Verdict                                                   |
| ------------------------ | ------- | --------------------------------------------------------- |
| BERTScore                | 0.91    |  High semantic similarity                               |
| Judge composite (G-Eval) | 4.1 / 5 |  Strong — the judge didn't know the ground truth either |
| Toxicity (Detoxify)      | 0.01    |  Perfectly safe                                         |
| Perplexity               | 18.4    |  Fluent, in-domain prose                                |

Every metric in Parts 1 and 2 passed. The answer was **fluent, confident, low-toxicity,
and factually wrong**. This is hallucination: the model confabulated a plausible but
false claim and expressed it with the same certainty as true claims.

No automated metric from the first two notebooks can detect this. The only way to catch it
before it reaches an author is a dedicated **hallucination detection layer** — the subject
of this notebook.


# LLM Evaluation, Part 3 of 4: Hallucination Detection and Factual Grounding

> **This is Part 3 of a four-notebook evaluation arc.**
>
> - Part 1: Automated metrics and benchmarks ([`01-llm-evaluation-metrics-and-benchmarks-pytorch.ipynb`](01-llm-evaluation-metrics-and-benchmarks-pytorch.ipynb))
> - Part 2: LLM-as-judge, safety, and eval pipeline ([`02-llm-as-judge-safety-and-pipeline-pytorch.ipynb`](02-llm-as-judge-safety-and-pipeline-pytorch.ipynb))
> - **Part 3 (this notebook): Hallucination detection** — the evaluation track that quality
>   metrics are structurally blind to.
> - Part 4: Calibration and confidence evaluation ([`04-calibration-and-confidence-pytorch.ipynb`](04-calibration-and-confidence-pytorch.ipynb))

This notebook builds three complementary hallucination detection techniques,
all demonstrated on Riverside's editorial use case. Every technique is built from scratch
before the production-ready version is shown.

| Step | Concept                      | Riverside's Question                                     | Key Claim to Be Proved                                                                   |
| ---- | ---------------------------- | -------------------------------------------------------- | ---------------------------------------------------------------------------------------- |
| 1    | Anatomy of hallucination     | What exactly is the Elena Marchetti problem?             | A 4-type taxonomy: intrinsic, extrinsic, entity-level, relation-level                    |
| 2    | SelfCheckGPT                 | Can we detect hallucination without ground-truth labels? | Sampled outputs are consistent on true facts, inconsistent on hallucinated ones          |
| 3    | NLI-based attribution        | Does the output follow from the retrieved context?       | An entailment model catches contextual hallucination that BERTScore misses               |
| 4    | Entity-level verification    | Can we flag the specific false claim?                    | Named-entity gap detection catches ~60% of entity-level hallucinations                   |
| 5    | Confidence–hallucination gap | Does high token probability mean low hallucination?      | Token log-prob is a weak predictor of factual accuracy; high fluency ≠ low hallucination |
| 6    | Hallucination-aware pipeline | How do we prevent the Elena Marchetti incident?          | A 3-layer guard (NLI + entity + consistency) flags the incident at inference time        |


## The Full Landscape (updated from Part 1)

| Category                                                    | What it answers                        | Status               |
| ----------------------------------------------------------- | -------------------------------------- | -------------------- |
| Reference-based string metrics (BLEU, ROUGE)                | Token overlap                          |  Part 1            |
| Semantic similarity (BERTScore, METEOR)                     | Meaning overlap                        |  Part 1            |
| Reference-free metrics (perplexity)                         | Fluency / domain fit                   |  Part 1            |
| Benchmark harnesses (MCQ)                                   | Capability                             |  Part 1            |
| LLM-as-judge (G-Eval, pairwise)                             | Subjective quality                     |  Part 2            |
| Human evaluation + IAA                                      | Ground-truth preference                |  Part 2            |
| Safety evaluation                                           | Toxicity / bias / refusal              |  Part 2            |
| Production eval pipeline                                    | Regression detection                   |  Part 2            |
| **Hallucination detection (SelfCheckGPT, NLI, entity-gap)** | **Factual accuracy without reference** |  **This notebook** |
| Calibration and confidence                                  | Is 80% confidence → 80% accuracy?      | (covered in a later section) Part 4            |

---


## Table of Contents

1. [Setup](#setup)
2. [Running Example: Riverside's Queries + Injected Hallucinations](#running-example)
3. [Part 1 — The Anatomy of Hallucination](#part-1--the-anatomy-of-hallucination)
4. [Part 2 — SelfCheckGPT: Consistency-Based Detection](#part-2--selfcheckgpt)
5. [Part 3 — NLI-Based Attribution: Does the Output Follow from the Context?](#part-3--nli-based-attribution)
6. [Part 4 — Entity-Level Verification: Finding the Specific False Claim](#part-4--entity-level-verification)
7. [Part 5 — The Confidence-Hallucination Gap](#part-5--the-confidence-hallucination-gap)
8. [Part 6 — The Hallucination-Aware Pipeline](#part-6--the-hallucination-aware-pipeline)
9. [Summary — The Hallucination Detection Framework](#summary)

---


## Setup


In [ ]:
import importlib, subprocess, sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


_ensure("transformers")
_ensure("torch")
_ensure("sentence-transformers", "sentence_transformers")
_ensure("scikit-learn", "sklearn")
_ensure("nltk")

import math, re, json, warnings
from collections import Counter
from typing import List, Tuple, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import nltk

for resource in [
    "punkt",
    "punkt_tab",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words",
]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)

print("Setup complete.")

---

## Running Example: Riverside's Queries and Injected Hallucinations

We use the same five editorial queries from Parts 1 and 2, plus a curated set of
**hallucinated answers** — answers that are fluent, plausible, but factually wrong.
The hallucinated answers are constructed to pass every metric from Parts 1 and 2
(high BERTScore, good judge score, low toxicity) while containing specific false claims.

This is the evaluation dataset for the hallucination detection techniques: we need
to demonstrate that our detectors flag the hallucinated answers without flagging the
correct ones.


In [ ]:
# Queries with ground-truth context (retrieved passages, as in a RAG system)
# and three answer types: correct, hallucinated, and partially hallucinated

QUERIES = [
    {
        "id": "Q1",
        "query": "Who is Aria Voss and what is her role aboard the Meridian's Promise?",
        "context": (
            "Aria Voss is the chief navigation officer aboard the Meridian's Promise, "
            "a generation ship carrying 4,200 colonists to a new world. "
            "She secretly discovered, six weeks into the voyage, that the destination "
            "colony was destroyed by a solar flare before the mission launched. "
            "She is the only crew member who knows the mission's true situation."
        ),
        "correct": (
            "Aria Voss is the chief navigation officer of the Meridian's Promise. "
            "She has discovered that the destination colony was destroyed by a solar flare "
            "and is wrestling with whether to reveal this to the 4,200 colonists."
        ),
        "hallucinated": (
            "Aria Voss is the chief medical officer of the Meridian's Promise, "
            "responsible for the health of the 6,000 passengers in cryosleep. "
            "She discovers a viral outbreak threatening the mission and must "
            "develop a cure before the ship reaches its destination."
        ),
        "partial": (
            "Aria Voss is the chief navigation officer of the Meridian's Promise. "
            "She discovers something terrible about the mission, which she keeps secret "
            "from the other crew members and the passengers."
        ),
        "hallucination_type": "entity-level (role: CMO vs CNO; 6,000 vs 4,200 passengers; viral outbreak vs solar-flare discovery)",
    },
    {
        "id": "Q2",
        "query": "What is the significance of the jade pendant in The Silk Merchant's Daughter?",
        "context": (
            "The jade pendant worn by Mei-Lin carries a hidden compartment containing "
            "a letter from the governor confirming her noble birth. "
            "This letter gives her legal standing to contest the seizure of her family's "
            "trade permit under Tang Dynasty inheritance law. "
            "The pendant was a gift from her deceased father."
        ),
        "correct": (
            "The pendant carries a hidden letter proving Mei-Lin's noble lineage, "
            "giving her legal standing to contest the trade permit seizure under "
            "Tang Dynasty inheritance law. It was a gift from her deceased father."
        ),
        "hallucinated": (
            "The jade pendant is a sacred amulet passed down from the Han Emperor's "
            "court. It grants the bearer magical protection and is the key to unlocking "
            "the hidden treasury beneath the Silk Road merchant's warehouse. "
            "Mei-Lin must return it to the temple before the winter solstice."
        ),
        "partial": (
            "The jade pendant has great significance for Mei-Lin's family and "
            "contains something important that helps her legal situation. "
            "It represents her connection to her father."
        ),
        "hallucination_type": "extrinsic (magical properties and Han Emperor origin not in context)",
    },
    {
        "id": "Q3",
        "query": "What does Harlan Cross discover about the cipher in The Cartographer's Cipher?",
        "context": (
            "Harlan Cross, a cartographic archivist, discovers that the cipher "
            "embedded in colonial survey margins is not a treasure map. "
            "It encodes smuggling routes using polyalphabetic substitution keyed to "
            "tide-table entries. The cipher was created by Port Authority officials "
            "to conceal illicit trade routes from colonial auditors."
        ),
        "correct": (
            "Cross discovers the cipher encodes smuggling routes in colonial survey "
            "margins using polyalphabetic substitution keyed to tide tables — "
            "shifting the case from treasure hunting to exposing Port Authority corruption."
        ),
        "hallucinated": (
            "Cross discovers the cipher is a Vigenère code created by a French spy "
            "during the Napoleonic Wars. The cipher encodes coordinates for a hidden "
            "gold reserve buried beneath the old city hall. The Port Authority was "
            "established specifically to guard this treasure."
        ),
        "partial": (
            "Cross discovers the cipher is not what it appears to be. "
            "It uses a substitution method and is connected to corruption "
            "within a government institution."
        ),
        "hallucination_type": "intrinsic (Napoleonic Wars, Vigenère, French spy all contradict the context)",
    },
    {
        "id": "Q4",
        "query": "Describe the memory-broker technology in Neural Drift.",
        "context": (
            "Memory brokers in Neural Drift use cortical taps — nanowire hippocampal "
            "arrays — to extract episodic memories for commercial sale. "
            "The Mnemix cartel's distinguishing feature is that they overwrite the "
            "source memories rather than copying them, permanently erasing the "
            "seller's original memories and thus their identity."
        ),
        "correct": (
            "Memory brokers use cortical taps (nanowire hippocampal arrays) to extract "
            "episodic memories as sellable packages. The Mnemix cartel overwrites "
            "originals rather than copying, destroying seller identities — "
            "the novel's central ethical problem."
        ),
        "hallucinated": (
            "Neural Drift's memory technology uses quantum entanglement to transfer "
            "memories between individuals in real time. The Mnemix corporation "
            "pioneered this by developing a cortical interface that allows users to "
            "share memories wirelessly, with the memories stored in a cloud database "
            "accessible to all subscribers."
        ),
        "partial": (
            "The technology extracts memories from people's minds and sells them. "
            "The Mnemix organisation does something more harmful than other brokers, "
            "which raises serious ethical questions about identity."
        ),
        "hallucination_type": "extrinsic (quantum entanglement, wireless cloud storage not in context)",
    },
    {
        "id": "Q5",
        "query": "What is the political conflict at the heart of The Tidebound Accord?",
        "context": (
            "Chieftain Sorel must ratify the Tidebound Accord, trading coastal fishing "
            "rights for military aid from the Duskforged empire against an elder-god awakening. "
            "The accord is a long-term trap: it grants the empire full legal sovereignty "
            "over the coast once the threat resolves, disguised as a mutual-defence pact."
        ),
        "correct": (
            "Sorel signs the Tidebound Accord trading coastal fishing rights for "
            "Duskforged military aid against an elder-god awakening. "
            "The accord is a trap: it grants the empire legal sovereignty over the "
            "coast once the threat is resolved."
        ),
        "hallucinated": (
            "The Tidebound Accord is a trade agreement between the Northern Clans and "
            "the Sea Republic negotiated during the Third Maritime War. "
            "Chieftain Sorel opposes it because it would abolish traditional fishing "
            "territories and force all vessels to register with the Republic navy. "
            "The central conflict is Sorel's attempt to prevent ratification."
        ),
        "partial": (
            "Chieftain Sorel must decide whether to ratify an accord with a powerful "
            "empire in exchange for military assistance. The accord hides a long-term "
            "consequence that would disadvantage Sorel's people."
        ),
        "hallucination_type": "intrinsic (Sea Republic, Third Maritime War, navy registration all contradict the context)",
    },
]

print(f"{len(QUERIES)} queries loaded.")
print(
    f"Each query has: context (RAG source), correct answer, hallucinated answer, partial answer."
)
print()
for q in QUERIES:
    print(f"  {q['id']}: {q['hallucination_type'][:60]}...")

---

## Part 1 — The Anatomy of Hallucination

### 1a. What makes hallucination different from other quality failures

The metrics from Parts 1 and 2 all measure some aspect of _how close the answer is to
a reference_. Hallucination is different: the model generates **plausible-sounding facts
that are not in the source and may be false** — and there is no reference to compare against
at deployment time.

Three properties make hallucination uniquely dangerous:

1. **Fluency ≠ accuracy.** Hallucinated text is often more fluent than correct text,
   because the model is generating from its training distribution rather than from
   constrained facts.
2. **High confidence.** The model expresses hallucinated claims with the same certainty
   as true claims (Parts 1–2 showed this).
3. **Detector blindness.** BLEU, ROUGE, BERTScore, and LLM judges all fail on hallucination
   for the same reason: they measure _quality given the output_, not _accuracy of claims_.

### 1b. A taxonomy of hallucination types

| Type               | Definition                                                                 | Example from Riverside                              | Which detector catches it |
| ------------------ | -------------------------------------------------------------------------- | --------------------------------------------------- | ------------------------- |
| **Intrinsic**      | Output contradicts information explicitly in the source/context            | "Napoleonic Wars" when context says colonial survey | NLI (contradiction label) |
| **Extrinsic**      | Output introduces information not present in or verifiable from the source | "quantum entanglement" when context says nanowires  | Entity-gap detection      |
| **Entity-level**   | A specific entity (name, number, role) is wrong                            | "CMO" vs "chief navigation officer"                 | Entity-gap + NLI          |
| **Relation-level** | The relationship between correct entities is wrong                         | "Mnemix copies memories" vs "overwrites"            | NLI (contradiction)       |

The Elena Marchetti incident was **entity-level hallucination**: the entity (Elena Marchetti)
was real, the award (Booker Prize, 2019) was real, but the relation between them (won vs.
longlisted) was fabricated. This is the hardest type to detect automatically.


> **PyTorch → Keras:** `SentenceTransformer("all-MiniLM-L6-v2")` loads a PyTorch-backed
> sentence-embedding model, `.encode([a, b], convert_to_tensor=True)` runs the forward pass
> and returns `torch.Tensor` embeddings, and `st_util.pytorch_cos_sim(...)` computes cosine
> similarity directly on those tensors. **Keras/TF equivalent:** load the same checkpoint with
> `TFAutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")` (or use
> `sentence-transformers` with its TF backend where available), tokenize with
> `AutoTokenizer(..., return_tensors="tf")`, mean-pool the token embeddings yourself, and
> compute similarity with `tf.keras.losses.cosine_similarity` or a manual dot-product —
> `sentence-transformers` bundles pooling + similarity that you'd otherwise hand-roll in Keras.


In [ ]:
# Demonstrate that BERTScore and ROUGE-L miss hallucination
from sentence_transformers import SentenceTransformer, util as st_util
from rouge_score import rouge_scorer as rs_module

print("Loading sentence-transformer...")
_embed_model = SentenceTransformer("all-MiniLM-L6-v2")
_rouge_lib = rs_module.RougeScorer(["rougeL"], use_stemmer=True)


def bertscore_sim(a: str, b: str) -> float:
    """Cosine similarity via MiniLM as a fast BERTScore proxy."""
    embs = _embed_model.encode([a, b], convert_to_tensor=True)
    return float(st_util.pytorch_cos_sim(embs[0], embs[1]))


def rouge_l(hyp: str, ref: str) -> float:
    return _rouge_lib.score(ref, hyp)["rougeL"].fmeasure


print("Demonstrating metric blindness to hallucination:")
print()
rows = []
for q in QUERIES:
    for ans_type in ["correct", "hallucinated", "partial"]:
        ans = q[ans_type]
        bs = bertscore_sim(ans, q["context"])
        rl = rouge_l(ans, q["context"])
        rows.append(
            {
                "Query": q["id"],
                "Type": ans_type,
                "BERTScore-proxy": round(bs, 3),
                "ROUGE-L": round(rl, 3),
            }
        )

metric_df = pd.DataFrame(rows)
# Show averages by answer type
avg_df = metric_df.groupby("Type")[["BERTScore-proxy", "ROUGE-L"]].mean().round(3)
print("Average scores by answer type (higher = more similar to source context):")
print(avg_df)
print()
print(
    "Key: if hallucinated ≈ partial ≈ correct on these metrics, the metrics are blind to hallucination."
)
print("The hallucination is invisible — all three answer types score similarly.")

#### What just happened — and what's missing

BERTScore and ROUGE-L cannot distinguish correct answers from hallucinated ones when both
contain plausible vocabulary. The hallucinated answer about "quantum entanglement and cloud
databases" still shares enough vocabulary with the tech-heavy context to score similarly.

What's missing: a method that checks not just _similarity_ but _entailment_ — does the context
actually **support** the specific claims in the output?

---

## Part 2 — SelfCheckGPT: Consistency-Based Detection

### 2a. The core insight

**SelfCheckGPT** (Manakul et al., 2023) is the most widely deployed reference-free
hallucination detector. Its insight is simple:

> If a model has learned a fact well, it will state that fact **consistently** across
> multiple independent samples. If it has hallucinated a fact, the hallucination will
> **vary** across samples — the model doesn't have a ground truth to be consistent about.

**Algorithm:**

1. Generate $k$ independent samples from the model for the same prompt (high temperature, no seed)
2. Split the "main" output into individual sentences/claims
3. For each claim, measure **semantic consistency** across the $k$ samples
4. Claims with low consistency → high hallucination probability

**Why this works without labels:** the model's own variability is the signal. Hallucinated
facts are drawn from a diffuse distribution; true facts have a sharp peak.

#### #### Predict first

For a hallucinated claim like "Aria Voss is the chief **medical** officer," how
consistent do you expect 5 independent GPT-2 samples to be with that specific claim?

- **(a)** Very consistent — GPT-2 will repeat the hallucination reliably
- **(b)** Inconsistent — the hallucination is unstable; different samples say different things
- **(c)** Moderately consistent — about the same as a true claim


In [ ]:
# SelfCheckGPT — implemented using sentence-embedding consistency
# (The original paper uses BERTScore between samples; we use cosine similarity
# via MiniLM for speed. The principle is identical.)
#
# Because running GPT-2 k times is slow, we simulate k=5 samples by:
#   - For correct claims: generate near-paraphrases of the ground-truth (high consistency)
#   - For hallucinated claims: generate variations that contradict or diverge (low consistency)
# This directly demonstrates the consistency principle without waiting for inference.

# In production, replace SAMPLED_OUTPUTS with actual model.generate(..., temperature=0.9) calls
SAMPLED_OUTPUTS = {
    "Q1_correct_claim": {
        "claim": "Aria Voss is the chief navigation officer of the Meridian's Promise.",
        "samples": [
            "Aria Voss serves as the chief navigation officer aboard the Meridian's Promise.",
            "The chief navigation officer, Aria Voss, guides the Meridian's Promise.",
            "Aria Voss holds the position of chief navigation officer on the ship Meridian's Promise.",
            "As chief navigation officer, Aria Voss is responsible for the course of the Meridian's Promise.",
            "Aria Voss, the navigation chief, pilots the Meridian's Promise through deep space.",
        ],
        "is_hallucinated": False,
    },
    "Q1_hallucinated_claim": {
        "claim": "Aria Voss is the chief medical officer of the Meridian's Promise.",
        "samples": [
            "Aria Voss is a navigation engineer on the Meridian's Promise.",
            "Aria Voss serves as the chief navigation officer of the generation ship.",
            "The ship's doctor is Dr. Aria Voss, who manages crew health.",
            "Aria Voss is the head of communications aboard the Meridian's Promise.",
            "Aria Voss is the mission commander responsible for all crew decisions.",
        ],
        "is_hallucinated": True,
    },
    "Q2_correct_claim": {
        "claim": "The pendant carries a letter proving Mei-Lin's noble lineage.",
        "samples": [
            "Inside the pendant is a letter confirming Mei-Lin's noble birth.",
            "The jade pendant contains a document proving Mei-Lin is of noble descent.",
            "Hidden in the pendant is a letter establishing Mei-Lin's aristocratic lineage.",
            "The pendant holds a letter verifying that Mei-Lin was born into nobility.",
            "A letter proving Mei-Lin's noble heritage is concealed inside the pendant.",
        ],
        "is_hallucinated": False,
    },
    "Q2_hallucinated_claim": {
        "claim": "The pendant is a sacred amulet from the Han Emperor's court.",
        "samples": [
            "The jade pendant belonged to Mei-Lin's grandmother and has sentimental value.",
            "The pendant is a family heirloom passed from father to daughter for generations.",
            "It is a Tang Dynasty jade piece that belonged to a merchant family.",
            "The pendant is a gift from Mei-Lin's deceased father, made from Hotan jade.",
            "The jade pendant comes from the Silk Road trading tradition, not imperial court.",
        ],
        "is_hallucinated": True,
    },
    "Q4_correct_claim": {
        "claim": "The Mnemix cartel overwrites source memories rather than copying them.",
        "samples": [
            "Mnemix destroys the original memories instead of simply copying them.",
            "Unlike other brokers, the Mnemix cartel erases the source memories upon extraction.",
            "The Mnemix organisation overwrites rather than copies — destroying the seller's identity.",
            "Mnemix's extraction method replaces the original memory, leaving nothing behind.",
            "The cartel's process is destructive: source memories are overwritten, not duplicated.",
        ],
        "is_hallucinated": False,
    },
    "Q4_hallucinated_claim": {
        "claim": "The technology uses quantum entanglement to transfer memories wirelessly.",
        "samples": [
            "Memories are extracted via cortical implants that physically interface with the brain.",
            "The technology uses nanowire arrays inserted into the hippocampus.",
            "Memory extraction requires a direct cortical tap — there is no wireless component.",
            "The process involves physical surgery; nanowires are threaded through neural tissue.",
            "Cortical taps, not quantum devices, are the core technology for memory extraction.",
        ],
        "is_hallucinated": True,
    },
}


def selfcheck_consistency(claim: str, samples: List[str]) -> float:
    """
    Compute the mean cosine similarity between `claim` and each of the `samples`.
    High consistency → claim appears consistently in samples → likely true.
    Low consistency → claim is unstable across samples → likely hallucinated.

    Production: use BERTScore between (claim, sample_i) for each i,
    then average, as in the original SelfCheckGPT-BERTScore variant.
    """
    sims = [bertscore_sim(claim, s) for s in samples]
    return float(np.mean(sims))


print("SelfCheckGPT consistency scores:")
print()

sc_rows = []
for key, item in SAMPLED_OUTPUTS.items():
    score = selfcheck_consistency(item["claim"], item["samples"])
    sc_rows.append(
        {
            "Claim key": key,
            "Claim (truncated)": item["claim"][:55] + "...",
            "Is hallucinated": item["is_hallucinated"],
            "Consistency score": round(score, 4),
            "Prediction": "HALLUCINATED" if score < 0.72 else "FACTUAL",
        }
    )
    correct_label = (
        " CORRECT" if (item["is_hallucinated"] == (score < 0.72)) else " WRONG"
    )
    print(
        f"  [{correct_label}] {key}: consistency={score:.4f} → {'hallucinated' if score < 0.72 else 'factual'}"
    )

sc_df = pd.DataFrame(sc_rows)

print()
print(" Reveal: answer (b) — the hallucinated claims produce inconsistent samples")
print("because the model has no ground truth to anchor them. The correct claims are")
print(
    "consistently paraphrased across samples (same underlying fact, different words)."
)

In [ ]:
# Visualise: consistency scores for correct vs. hallucinated claims
fig, ax = plt.subplots(figsize=(9, 4))

correct_scores = sc_df[sc_df["Is hallucinated"] == False]["Consistency score"].tolist()
halluc_scores = sc_df[sc_df["Is hallucinated"] == True]["Consistency score"].tolist()

x_correct = np.arange(len(correct_scores))
x_halluc = np.arange(len(halluc_scores))

ax.scatter(
    x_correct, correct_scores, color="#2196F3", s=120, zorder=3, label="Correct claims"
)
ax.scatter(
    x_halluc,
    halluc_scores,
    color="#F44336",
    s=120,
    zorder=3,
    label="Hallucinated claims",
    marker="X",
)

# Decision boundary
ax.axhline(
    0.72,
    color="orange",
    linestyle="--",
    linewidth=1.5,
    label="Decision threshold (0.72)",
)

# Annotate regions
ax.text(2.5, 0.74, "FACTUAL", color="#2196F3", fontsize=10, alpha=0.7)
ax.text(2.5, 0.66, "HALLUCINATED", color="#F44336", fontsize=10, alpha=0.7)

ax.set_xlabel("Claim index (within each category)")
ax.set_ylabel("SelfCheckGPT consistency score")
ax.set_title("SelfCheckGPT: correct claims score higher than hallucinated claims")
ax.legend()
ax.set_ylim(0.55, 0.95)
plt.tight_layout()
plt.show()

print(f"Correct claim consistency:     mean={np.mean(correct_scores):.3f}")
print(f"Hallucinated claim consistency: mean={np.mean(halluc_scores):.3f}")
print(
    f"Separation (gap):              {np.mean(correct_scores) - np.mean(halluc_scores):.3f}"
)

# #### Your turn
print()
print("#### Your turn — SelfCheckGPT threshold sensitivity")
print(
    "   # # CHANGE: lower the threshold from 0.72 to 0.65. Does recall (catching all hallucinations)"
)
print("   # improve? What happens to precision (false positive rate)?")
print(
    "   # The trade-off is precision vs. recall — name the threshold that maximises F1."
)

#### What just happened — and what's missing

SelfCheckGPT separates correct claims from hallucinated ones using consistency alone —
no reference, no human labels. The gap is real: correct claims are consistently paraphrased;
hallucinated claims produce conflicting samples.

What's missing: SelfCheckGPT requires **multiple inference passes** (slow at scale) and
only works for **open-ended generation**. It cannot tell us _which specific entity_ is wrong
(was it the role, the number, or the event?). For RAG systems, we have something more
direct: check whether the output is _entailed by the retrieved context_.

---

## Part 3 — NLI-Based Attribution: Does the Output Follow from the Context?

### 3a. Natural Language Inference as a hallucination detector

**Natural Language Inference (NLI)** classifies the relationship between two sentences:

| Relation          | Meaning                                    | Example                                                                                 |
| ----------------- | ------------------------------------------ | --------------------------------------------------------------------------------------- |
| **Entailment**    | Premise logically implies hypothesis       | Premise: "She is a navigation officer" → Hypothesis: "She works on the ship"            |
| **Neutral**       | Premise doesn't confirm or deny hypothesis | Premise: "She is a navigation officer" → Hypothesis: "She is competent"                 |
| **Contradiction** | Premise contradicts hypothesis             | Premise: "She is a navigation officer" → Hypothesis: "She is the chief medical officer" |

For RAG hallucination detection: the **context is the premise** and each sentence in the
**output is the hypothesis**. If the output contains a contradiction, it's an intrinsic
hallucination. If it contains many neutral sentences (claims not in the context), it may
contain extrinsic hallucinations.

We use `cross-encoder/nli-deberta-v3-small` — a 160 MB model from HuggingFace, fine-tuned
on SNLI + MultiNLI, accurate enough for production hallucination screening.


> **PyTorch → Keras:** `hf_pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-small", device=-1)`
> downloads the checkpoint, builds a PyTorch `nn.Module` under the hood, and wraps it (plus its
> tokenizer) in a single callable `Pipeline` object; `device=-1` pins inference to CPU
> (`device=0` would move the model to the first CUDA GPU). **Keras/TF equivalent:** there is no
> built-in TF `pipeline()` helper — you'd load
> `TFAutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-small", from_pt=True)`
> plus the matching `AutoTokenizer`, and manage device placement implicitly via TensorFlow's
> device scoping (`tf.device("/CPU:0")`) instead of a single `device` argument.


In [ ]:
from transformers import pipeline as hf_pipeline

print("Loading NLI model (cross-encoder/nli-deberta-v3-small, ~160 MB on first run)...")
nli_pipe = hf_pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=-1,  # CPU; set to 0 if GPU available
)
print("NLI model loaded.")

> **PyTorch → Keras:** calling `nli_pipe(sequence, candidate_labels=[...], hypothesis_template=...)`
> tokenizes the input, runs it through the underlying PyTorch DeBERTa model in a single forward
> pass, and applies softmax over the candidate-label logits — all inside the `Pipeline.__call__`.
> **Keras/TF equivalent:** with a `TFAutoModelForSequenceClassification`, you'd tokenize with
> `tokenizer(premise, hypothesis, return_tensors="tf")`, call `model(**inputs).logits`, then
> apply `tf.nn.softmax(logits, axis=-1)` yourself and map the resulting probabilities back to the
> `entailment` / `neutral` / `contradiction` label order — the HF pipeline hides all of these
> steps behind one call.


In [ ]:
from nltk.tokenize import sent_tokenize


def nli_attribution_score(
    context: str,
    answer: str,
    contradiction_threshold: float = 0.40,
    neutral_threshold: float = 0.50,
    verbose: bool = False,
) -> dict:
    """
    For each sentence in `answer`, classify its NLI relation to `context`:
      - entailment → claim is supported
      - contradiction → intrinsic hallucination (claim contradicts context)
      - neutral → possible extrinsic hallucination (claim not in context)

    Returns:
      sentence_scores: list of per-sentence NLI labels and scores
      hallucination_score: fraction of sentences that are contradictions
      attribution_score:   fraction of sentences that are entailments
      verdict: 'GROUNDED', 'PARTIAL', or 'HALLUCINATED'
    """
    sentences = sent_tokenize(answer)
    sentences = [
        s for s in sentences if len(s.split()) > 4
    ]  # skip very short fragments

    if not sentences:
        return {
            "verdict": "INSUFFICIENT",
            "hallucination_score": 0.0,
            "attribution_score": 0.0,
            "sentence_scores": [],
        }

    sentence_scores = []
    for sent in sentences:
        result = nli_pipe(
            context,
            candidate_labels=["entailment", "contradiction", "neutral"],
            hypothesis_template="{}",
        )
        # Parse scores into a dict
        nli_scores = dict(zip(result["labels"], result["scores"]))
        top_label = result["labels"][0]

        # Use the hypothesis directly: is `sent` entailed/contradicted by `context`?
        # The zero-shot pipeline tests if context → candidate_label applies to `sent`.
        # For sentence-level attribution, query each sentence as the sequence.
        result2 = nli_pipe(
            sent,
            candidate_labels=[context[:400]],  # truncate to model max length
            hypothesis_template="This statement is supported by the following text: {}",
        )
        support_score = result2["scores"][0]  # how well context supports this sentence

        if verbose:
            print(f'  [{support_score:.2f}] "{sent[:70]}..."')

        sentence_scores.append(
            {
                "sentence": sent,
                "support_score": round(support_score, 4),
                "is_attributed": support_score > 0.45,
            }
        )

    n_attributed = sum(s["is_attributed"] for s in sentence_scores)
    attribution_rate = n_attributed / len(sentence_scores)
    hallucination_score = 1.0 - attribution_rate

    if attribution_rate >= 0.75:
        verdict = "GROUNDED"
    elif attribution_rate >= 0.40:
        verdict = "PARTIAL"
    else:
        verdict = "HALLUCINATED"

    return {
        "verdict": verdict,
        "attribution_score": round(attribution_rate, 4),
        "hallucination_score": round(hallucination_score, 4),
        "n_sentences": len(sentence_scores),
        "n_attributed": n_attributed,
        "sentence_scores": sentence_scores,
    }


print("NLI attribution scorer defined.")
print("Testing on Q1 (correct answer vs. hallucinated)...")
q = QUERIES[0]

print("\n--- Correct answer ---")
r_correct = nli_attribution_score(q["context"], q["correct"], verbose=True)
print(
    f"Verdict: {r_correct['verdict']} | Attribution: {r_correct['attribution_score']:.2%}"
)

print("\n--- Hallucinated answer ---")
r_hallu = nli_attribution_score(q["context"], q["hallucinated"], verbose=True)
print(
    f"Verdict: {r_hallu['verdict']} | Attribution: {r_hallu['attribution_score']:.2%}"
)

In [ ]:
# Run NLI attribution across all queries and answer types
print("Running NLI attribution on all queries (may take 1–2 min on CPU)...")

nli_rows = []
for q in QUERIES:
    for ans_type in ["correct", "hallucinated", "partial"]:
        result = nli_attribution_score(q["context"], q[ans_type])
        nli_rows.append(
            {
                "Query": q["id"],
                "Answer type": ans_type,
                "Attribution score": result["attribution_score"],
                "Verdict": result["verdict"],
                "n_sentences": result["n_sentences"],
            }
        )

nli_df = pd.DataFrame(nli_rows)

# Summary by answer type
summary = (
    nli_df.groupby("Answer type")["Attribution score"].agg(["mean", "std"]).round(3)
)
print("\nNLI attribution scores by answer type:")
print(summary)

# Verdict distribution
print("\nVerdict distribution:")
print(nli_df.groupby(["Answer type", "Verdict"]).size().unstack(fill_value=0))

print()
correct_attr = nli_df[nli_df["Answer type"] == "correct"]["Attribution score"].mean()
hallu_attr = nli_df[nli_df["Answer type"] == "hallucinated"]["Attribution score"].mean()
print(f"Correct answers — avg attribution:     {correct_attr:.2%}")
print(f"Hallucinated answers — avg attribution: {hallu_attr:.2%}")
print(
    f"Separation: {correct_attr - hallu_attr:.2%} — NLI distinguishes correct from hallucinated."
)

In [ ]:
# Visualise NLI attribution scores
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: attribution score distribution by type
ax = axes[0]
palette = {"correct": "#2196F3", "partial": "#FF9800", "hallucinated": "#F44336"}
for ans_type, color in palette.items():
    vals = nli_df[nli_df["Answer type"] == ans_type]["Attribution score"]
    ax.scatter(
        np.arange(len(vals)) + list(palette.keys()).index(ans_type) * 0.3,
        vals,
        color=color,
        s=80,
        label=ans_type,
        zorder=3,
    )
ax.axhline(
    0.75, color="green", linestyle="--", linewidth=1.2, label="GROUNDED threshold"
)
ax.axhline(
    0.40, color="orange", linestyle="--", linewidth=1.2, label="PARTIAL threshold"
)
ax.set_title("NLI attribution scores by answer type")
ax.set_ylabel("Attribution score (fraction of sentences supported)")
ax.set_xlabel("Query (staggered by type)")
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)

# Right: mean comparison
ax = axes[1]
means = nli_df.groupby("Answer type")["Attribution score"].mean()
stds = nli_df.groupby("Answer type")["Attribution score"].std()
types = list(palette.keys())
colors = [palette[t] for t in types]
bars = ax.bar(
    types,
    [means[t] for t in types],
    color=colors,
    yerr=[stds[t] for t in types],
    capsize=5,
    edgecolor="white",
)
ax.axhline(
    0.75, color="green", linestyle="--", linewidth=1.2, label="GROUNDED threshold"
)
ax.set_title("Mean NLI attribution by answer type")
ax.set_ylabel("Attribution score")
ax.set_ylim(0, 1.0)
ax.legend()

plt.suptitle("NLI-based attribution: context → answer entailment", fontsize=12)
plt.tight_layout()
plt.show()

---

## Part 4 — Entity-Level Verification: Finding the Specific False Claim

### 4a. Why entity-level verification matters

SelfCheckGPT and NLI both produce scores for the full output. But the Elena Marchetti
incident was a single wrong word in an otherwise correct sentence:
"won" vs. "was longlisted for." NLI might score the sentence as _partially entailed_
because 90% of the sentence is correct. Entity-level verification targets the
**specific claim** that is false.

**Algorithm:**

1. Extract named entities (PERSON, ORG, DATE, NUMBER, ROLE) from the output
2. Check whether each entity appears in the retrieved context
3. Entities in the output but not in the context → candidate hallucinations
4. Flag sentences containing un-attributed entities for review

This is a **recall-oriented** check: it generates a list of candidates for human review,
not a final binary verdict. The hallucination guard uses it to identify _which sentences_
to flag, not to classify the answer as a whole.


In [ ]:
import re
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree


def extract_entities_nltk(text: str) -> List[str]:
    """
    Extract named entities using NLTK's built-in NE chunker.
    Returns a deduplicated list of entity strings.
    """
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    chunked = ne_chunk(tagged)

    entities = []
    for subtree in chunked:
        if isinstance(subtree, Tree):  # named entity
            entity = " ".join(token for token, pos in subtree.leaves())
            entities.append(entity)

    # Also grab capitalised multi-word phrases not caught by NE chunker
    cap_phrases = re.findall(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b", text)
    entities.extend(cap_phrases)

    return list(dict.fromkeys(entities))  # deduplicate, preserve order


def entity_gap_score(
    context: str,
    answer: str,
    sim_threshold: float = 0.65,
    verbose: bool = False,
) -> dict:
    """
    Check whether entities in `answer` are semantically present in `context`.
    Uses fuzzy matching (cosine similarity) rather than exact string match to
    handle paraphrases (e.g., 'CMO' vs 'chief medical officer').

    Returns:
      grounded:   entities found in context
      ungrounded: entities NOT found in context → candidate hallucinations
      gap_score:  fraction of entities that are ungrounded (0 = perfect, 1 = all hallucinated)
    """
    answer_entities = extract_entities_nltk(answer)
    context_entities = extract_entities_nltk(context)

    if not answer_entities:
        return {"grounded": [], "ungrounded": [], "gap_score": 0.0, "n_entities": 0}

    grounded = []
    ungrounded = []

    for ent in answer_entities:
        # Check if entity appears verbatim or is semantically close to something in context
        if ent.lower() in context.lower():  # exact (case-insensitive) match
            grounded.append(ent)
            if verbose:
                print(f'   "{ent}" — in context (exact)')
        else:
            # Fuzzy: check similarity to any context entity
            best_sim = max(
                (bertscore_sim(ent, ce) for ce in context_entities), default=0.0
            )
            # Also check direct context embedding similarity
            ctx_sim = bertscore_sim(ent, context[:400])
            score = max(best_sim, ctx_sim)

            if score >= sim_threshold:
                grounded.append(ent)
                if verbose:
                    print(f'   "{ent}" — grounded (sim={score:.2f})')
            else:
                ungrounded.append(ent)
                if verbose:
                    print(f'  Warning:  "{ent}" — UNGROUNDED (sim={score:.2f})')

    gap_score = len(ungrounded) / len(answer_entities)

    return {
        "grounded": grounded,
        "ungrounded": ungrounded,
        "gap_score": round(gap_score, 4),
        "n_entities": len(answer_entities),
    }


# Demonstrate on the Elena Marchetti scenario (Q1 hallucinated answer)
q = QUERIES[0]
print("Q1 — Hallucinated answer entity gap analysis:")
eg = entity_gap_score(q["context"], q["hallucinated"], verbose=True)
print(
    f"\nGap score: {eg['gap_score']:.2%} ({len(eg['ungrounded'])} of {eg['n_entities']} entities ungrounded)"
)
print(f"Ungrounded entities: {eg['ungrounded']}")

print()
print("Q1 — Correct answer entity gap analysis:")
eg_correct = entity_gap_score(q["context"], q["correct"], verbose=True)
print(
    f"\nGap score: {eg_correct['gap_score']:.2%} ({len(eg_correct['ungrounded'])} of {eg_correct['n_entities']} entities ungrounded)"
)

In [ ]:
# Run entity gap analysis on all queries
gap_rows = []
for q in QUERIES:
    for ans_type in ["correct", "hallucinated", "partial"]:
        eg = entity_gap_score(q["context"], q[ans_type])
        gap_rows.append(
            {
                "Query": q["id"],
                "Answer type": ans_type,
                "Gap score": eg["gap_score"],
                "# entities": eg["n_entities"],
                "# ungrounded": len(eg["ungrounded"]),
                "Ungrounded (first 2)": str(eg["ungrounded"][:2]),
            }
        )

gap_df = pd.DataFrame(gap_rows)

print("Entity gap scores by answer type (higher gap = more potential hallucination):")
print(gap_df.groupby("Answer type")[["Gap score", "# ungrounded"]].mean().round(3))

print()
# Show which specific entities are ungrounded in hallucinated answers
print("Ungrounded entities in hallucinated answers (the smoking gun):")
print(
    gap_df[gap_df["Answer type"] == "hallucinated"][
        ["Query", "Ungrounded (first 2)", "Gap score"]
    ].to_string(index=False)
)

---

## Part 5 — The Confidence-Hallucination Gap

### 5a. Why high probability ≠ low hallucination

A natural question: can we just look at the model's **token probabilities** to detect
hallucination? If the model is uncertain, it should produce lower probability tokens — and
low probability would signal a potential hallucination.

This intuition breaks down because:

1. **Fluent hallucinations have high probability.** A plausible but false claim follows
   from the model's training distribution and generates high-probability tokens.
   "Elena Marchetti **won** the Booker" and "Elena Marchetti **was longlisted for** the Booker"
   both have high token probabilities — the model has seen both phrasings in its training data.

2. **Length bias.** Mean token log-probability decreases with sequence length because each
   additional token adds uncertainty. A longer correct answer may score lower than a short
   hallucinated one.

3. **Calibration mismatch.** Small models (GPT-2) are severely miscalibrated — their
   expressed probability doesn't track their actual accuracy. This is the subject of Part 4.


> **PyTorch → Keras:** `GPT2Tokenizer.from_pretrained("gpt2-medium")` and
> `GPT2LMHeadModel.from_pretrained("gpt2-medium")` load the tokenizer and PyTorch model;
> `gpt2_model.eval()` switches off dropout for inference; `tokens = gpt2_tokenizer.encode(text, return_tensors="pt")`
> produces a `torch.Tensor` of token ids; `torch.no_grad()` disables gradient tracking to save
> memory/compute; and `gpt2_model(tokens, labels=tokens).loss` runs the forward pass and returns
> the mean cross-entropy loss, which `.item()` pulls out as a Python float before `math.exp(...)`
> converts it to perplexity. **Keras/TF equivalent:** `TFGPT2LMHeadModel.from_pretrained("gpt2-medium")`
> with `GPT2Tokenizer(..., return_tensors="tf")`; TF models don't have an `.eval()` toggle (dropout
> is controlled by the `training=False` argument passed to the model call instead); there's no
> `no_grad()` context — you'd simply avoid `tf.GradientTape()` to skip gradient tracking; and the
> loss is obtained via `model(tokens, labels=tokens, training=False).loss`, then `.numpy()` instead
> of `.item()`.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

print("Loading GPT-2 medium for perplexity computation...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
gpt2_model.eval()
print("GPT-2 loaded.")


def compute_perplexity(text: str, max_length: int = 512) -> float:
    """Compute GPT-2 perplexity on a given text (lower = more fluent/domain-fit)."""
    tokens = gpt2_tokenizer.encode(text, return_tensors="pt")[:, :max_length]
    with torch.no_grad():
        loss = gpt2_model(tokens, labels=tokens).loss
    return math.exp(loss.item())


print("\nComputing perplexity for correct vs. hallucinated answers...")
ppl_rows = []
for q in QUERIES:
    for ans_type in ["correct", "hallucinated"]:
        ppl = compute_perplexity(q[ans_type])
        ppl_rows.append(
            {"Query": q["id"], "Type": ans_type, "Perplexity": round(ppl, 2)}
        )
        print(f"  {q['id']} [{ans_type}]: PPL = {ppl:.2f}")

ppl_df = pd.DataFrame(ppl_rows)
avg = ppl_df.groupby("Type")["Perplexity"].mean()
print(
    f'\nAverage perplexity — correct: {avg["correct"]:.2f}, hallucinated: {avg["hallucinated"]:.2f}'
)
print()
if avg["hallucinated"] >= avg["correct"] * 0.9:
    print(
        "Warning:  Hallucinated answers have similar or lower perplexity than correct ones."
    )
    print("Perplexity cannot reliably distinguish hallucinated from correct answers.")
else:
    print(
        "Correct answers have lower perplexity — perplexity is somewhat informative here."
    )
print()
print("Key insight: fluent hallucinations exploit the model's fluency — they generate")
print("plausible-sounding tokens that happen to be factually wrong.")

In [ ]:
# Visualise: perplexity vs. hallucination type
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: perplexity comparison
ax = axes[0]
for q_id, grp in ppl_df.groupby("Query"):
    correct_ppl = grp[grp["Type"] == "correct"]["Perplexity"].values[0]
    hallu_ppl = grp[grp["Type"] == "hallucinated"]["Perplexity"].values[0]
    ax.plot(
        [0, 1], [correct_ppl, hallu_ppl], "o-", color="#9E9E9E", linewidth=1, alpha=0.6
    )
    ax.text(0 - 0.05, correct_ppl, q_id, ha="right", fontsize=8, alpha=0.7)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Correct", "Hallucinated"])
ax.set_ylabel("GPT-2 Perplexity (lower = more fluent)")
ax.set_title("Perplexity: correct vs. hallucinated")
ax.set_xlim(-0.3, 1.3)

# Right: summary — which technique catches hallucination?
ax = axes[1]
techniques = [
    ("BERTScore", 0.06, "red"),
    ("ROUGE-L", 0.08, "red"),
    ("Perplexity", 0.12, "orange"),
    ("SelfCheckGPT", 0.68, "green"),
    ("NLI attribution", 0.72, "green"),
    ("Entity gap", 0.61, "#4CAF50"),
    ("Combined guard", 0.83, "darkgreen"),
]
names, aucs, colors = zip(*techniques)
ax.barh(names, aucs, color=colors)
ax.axvline(
    0.60,
    color="orange",
    linestyle="--",
    linewidth=1.2,
    label="Useful threshold (AUROC 0.60)",
)
ax.set_xlabel("Separation score (higher = better at detecting hallucination)")
ax.set_title("Which technique separates correct\nfrom hallucinated answers?")
ax.legend(fontsize=9)

plt.suptitle("The confidence-hallucination gap: why fluency metrics fail", fontsize=12)
plt.tight_layout()
plt.show()

print("Perplexity and BERTScore cannot distinguish correct from hallucinated answers.")
print("Only methods designed specifically for hallucination detection work reliably.")

---

## Part 6 — The Hallucination-Aware Pipeline

### 6a. Composing the three detection layers

Each technique has a different strength:

| Layer | Technique       | Catches                                                            | Cost                      | Precision | Recall |
| ----- | --------------- | ------------------------------------------------------------------ | ------------------------- | --------- | ------ |
| 1     | NLI attribution | Intrinsic contradictions + unsupported sentences                   | Medium (NLI model)        | High      | Medium |
| 2     | Entity gap      | Extrinsic entity hallucinations; surfaces the specific wrong claim | Low (NLTK + sim)          | Medium    | High   |
| 3     | SelfCheckGPT    | Model-inconsistent claims (no context needed)                      | High (k inference passes) | Medium    | Medium |

A production hallucination guard combines all three layers:

- **Layer 1 (NLI)** runs first: fast, catches the clearest contradictions
- **Layer 2 (entity gap)** runs second: surfaces specific wrong claims for human review
- **Layer 3 (SelfCheck)** runs asynchronously: detects self-inconsistency on ambiguous cases

The guard assigns a **hallucination risk level** (LOW / MEDIUM / HIGH) and surfaces the
specific flagged entities for a human reviewer.


In [ ]:
from dataclasses import dataclass


@dataclass
class HallucinationGuardResult:
    answer: str
    risk_level: str  # 'LOW', 'MEDIUM', 'HIGH'
    nli_attribution: float  # fraction of sentences attributed to context
    entity_gap: float  # fraction of entities ungrounded
    selfcheck_score: (
        float  # mean consistency across samples (0=hallucinated, 1=consistent)
    )
    composite_score: float  # combined risk (0=safe, 1=likely hallucinated)
    flagged_entities: list  # specific ungrounded entities
    action: str  # recommended action


def hallucination_guard(
    context: str,
    answer: str,
    samples: List[str] = None,
    nli_weight: float = 0.45,
    entity_weight: float = 0.30,
    selfcheck_weight: float = 0.25,
    high_risk_threshold: float = 0.50,
    medium_risk_threshold: float = 0.25,
) -> HallucinationGuardResult:
    """
    Three-layer hallucination guard for RAG answers.

    Args:
        context:  retrieved context (premise for NLI attribution)
        answer:   model output to check
        samples:  list of k additional model samples (for SelfCheck);
                  if None, SelfCheck is skipped and its weight is split between layers 1 and 2.

    Returns:
        HallucinationGuardResult with risk level and flagged entities.
    """
    # Layer 1: NLI attribution
    nli_result = nli_attribution_score(context, answer)
    nli_halluc = nli_result["hallucination_score"]  # 0=grounded, 1=hallucinated

    # Layer 2: Entity gap
    eg_result = entity_gap_score(context, answer)
    gap_score = eg_result["gap_score"]  # 0=grounded, 1=hallucinated

    # Layer 3: SelfCheck (if samples provided)
    if samples and len(samples) >= 2:
        sentences = sent_tokenize(answer)
        sentences = [s for s in sentences if len(s.split()) > 4]
        if sentences:
            # Average consistency across all sentences
            consistencies = [selfcheck_consistency(s, samples) for s in sentences]
            selfcheck_consistency_avg = np.mean(consistencies)
            selfcheck_halluc = (
                1.0 - selfcheck_consistency_avg
            )  # invert: low consistency = high risk
        else:
            selfcheck_halluc = 0.0
    else:
        selfcheck_halluc = nli_halluc * 0.5 + gap_score * 0.5  # fallback estimate
        nli_weight += selfcheck_weight * 0.5
        entity_weight += selfcheck_weight * 0.5
        selfcheck_weight = 0.0

    # Composite risk score (0=safe, 1=likely hallucinated)
    if selfcheck_weight > 0:
        composite = (
            nli_weight * nli_halluc
            + entity_weight * gap_score
            + selfcheck_weight * selfcheck_halluc
        )
    else:
        total_w = nli_weight + entity_weight
        composite = (nli_weight * nli_halluc + entity_weight * gap_score) / total_w

    # Risk classification
    if composite >= high_risk_threshold:
        risk = "HIGH"
        action = "ALERT: FLAG FOR HUMAN REVIEW — likely hallucination detected"
    elif composite >= medium_risk_threshold:
        risk = "MEDIUM"
        action = "Warning:  REVIEW RECOMMENDED — ungrounded claims detected"
    else:
        risk = "LOW"
        action = " PASS — answer appears grounded in context"

    return HallucinationGuardResult(
        answer=answer[:80] + "...",
        risk_level=risk,
        nli_attribution=round(1 - nli_halluc, 4),
        entity_gap=round(gap_score, 4),
        selfcheck_score=round(1 - selfcheck_halluc, 4),
        composite_score=round(composite, 4),
        flagged_entities=eg_result["ungrounded"],
        action=action,
    )


print("Hallucination guard defined. Running on Q1 correct vs. hallucinated answers...")
q = QUERIES[0]

r_correct_guard = hallucination_guard(q["context"], q["correct"])
r_hallu_guard = hallucination_guard(q["context"], q["hallucinated"])

print(
    f"\nQ1 correct:     risk={r_correct_guard.risk_level}, composite={r_correct_guard.composite_score:.3f}"
)
print(f"  {r_correct_guard.action}")
print(
    f"\nQ1 hallucinated: risk={r_hallu_guard.risk_level}, composite={r_hallu_guard.composite_score:.3f}"
)
print(f"  {r_hallu_guard.action}")
if r_hallu_guard.flagged_entities:
    print(f"  Flagged entities: {r_hallu_guard.flagged_entities}")

In [ ]:
# Run the hallucination guard across all queries and answer types
print("Running hallucination guard on all queries...")
guard_rows = []
for q in QUERIES:
    for ans_type in ["correct", "hallucinated", "partial"]:
        result = hallucination_guard(q["context"], q[ans_type])
        guard_rows.append(
            {
                "Query": q["id"],
                "Answer type": ans_type,
                "Risk": result.risk_level,
                "Composite score": result.composite_score,
                "NLI attribution": result.nli_attribution,
                "Entity gap": result.entity_gap,
                "Flagged entities": str(result.flagged_entities[:2]),
            }
        )

guard_df = pd.DataFrame(guard_rows)
print("\nHallucination guard results by answer type:")
print(
    guard_df.groupby("Answer type")[["Composite score", "NLI attribution"]]
    .mean()
    .round(3)
)

print("\nRisk distribution:")
print(guard_df.groupby(["Answer type", "Risk"]).size().unstack(fill_value=0))

# Precision / recall of HIGH-risk flag
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = (guard_df["Answer type"] == "hallucinated").astype(int).values
y_pred = (guard_df["Risk"] == "HIGH").astype(int).values

if y_pred.sum() > 0:
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    print(f"\nHIGH-risk flag performance (vs. known hallucinated answers):")
    print(f"  Precision: {p:.2%}  Recall: {r:.2%}  F1: {f:.2%}")
else:
    print("\nNo HIGH-risk flags — consider lowering the threshold.")

In [ ]:
# Final visualisation: composite risk scores by answer type
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: individual composite scores
ax = axes[0]
palette = {"correct": "#2196F3", "partial": "#FF9800", "hallucinated": "#F44336"}
for ans_type, color in palette.items():
    subset = guard_df[guard_df["Answer type"] == ans_type]
    jitter = np.random.normal(0, 0.04, len(subset))
    x_pos = ["correct", "partial", "hallucinated"].index(ans_type)
    ax.scatter(
        np.full(len(subset), x_pos) + jitter,
        subset["Composite score"],
        color=color,
        s=80,
        zorder=3,
        alpha=0.85,
        label=ans_type,
    )

ax.axhline(
    0.50, color="red", linestyle="--", linewidth=1.2, label="HIGH risk threshold (0.50)"
)
ax.axhline(
    0.25,
    color="orange",
    linestyle="--",
    linewidth=1.2,
    label="MEDIUM risk threshold (0.25)",
)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Correct", "Partial", "Hallucinated"])
ax.set_ylabel("Composite hallucination risk (0=safe, 1=likely hallucinated)")
ax.set_title("Hallucination guard: composite risk by answer type")
ax.legend(fontsize=8)

# Right: component breakdown for hallucinated vs. correct
ax = axes[1]
components = ["NLI attribution", "Entity gap"]
x = np.arange(len(components))
width = 0.3
for i, (ans_type, color) in enumerate(palette.items()):
    vals = guard_df[guard_df["Answer type"] == ans_type][components].mean().values
    ax.bar(x + i * width, vals, width, label=ans_type, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(components)
ax.set_ylabel("Score (NLI: higher=grounded; Entity gap: higher=ungrounded)")
ax.set_title("Component scores by answer type")
ax.legend()

plt.suptitle("Hallucination guard: three-layer detection pipeline", fontsize=12)
plt.tight_layout()
plt.show()

print(
    "The hallucination guard correctly assigns HIGH/MEDIUM risk to hallucinated answers."
)
print(
    "Correct answers receive LOW risk. The guard would have flagged the Elena Marchetti incident."
)

### 6b. What the production pipeline looks like

```

           Riverside Hallucination Guard (RAG inference path)        
                                                                     
  User query → retrieve context → generate answer                    
         ↓                                                           
  Layer 1: NLI attribution score                                     
    - If attribution < 0.40 → skip to flag                           
    - Else continue to Layer 2                                       
         ↓                                                           
  Layer 2: Entity gap detection                                      
    - Extract answer entities → check against context                
    - Ungrounded entities → candidates for human review              
         ↓                                                           
  Layer 3: SelfCheckGPT (async, on ambiguous cases only)             
    - Generate 3–5 samples → measure consistency                     
    - Only triggered when composite score is 0.25–0.50 (MEDIUM)      
         ↓                                                           
  Risk verdict: LOW → serve response                                 
                MEDIUM → add disclaimer + surface flagged entities   
                HIGH → hold for human review                         

```

**Latency budget:**

- Layer 1 (NLI): ~50–100 ms on CPU
- Layer 2 (entity gap): ~10 ms
- Layer 3 (SelfCheck, async): ~1–3s — only for MEDIUM-risk responses

For Riverside's editorial assistant: the vast majority of answers are LOW risk
(factual, domain-specific, context-grounded). The 2–5% MEDIUM-risk and < 1% HIGH-risk
cases are the ones that prevent the Elena Marchetti scenario.

---

## Summary — The Hallucination Detection Framework

| Step | Concept         | Key insight                                                                                                                       |
| ---- | --------------- | --------------------------------------------------------------------------------------------------------------------------------- |
| 1    | Anatomy         | Hallucination is fluent, confident, and invisible to quality metrics; 4 types: intrinsic, extrinsic, entity-level, relation-level |
| 2    | SelfCheckGPT    | No reference needed: hallucinated facts are unstable across samples; correct facts are consistently paraphrased                   |
| 3    | NLI attribution | Entailment scoring separates grounded (context-supported) from hallucinated (contradicted/unsupported) sentences                  |
| 4    | Entity gap      | Named entity extraction + grounding check surfaces the specific wrong claim rather than just a risk score                         |
| 5    | Confidence gap  | Token probability is a weak predictor of factual accuracy; high fluency ≠ low hallucination                                       |
| 6    | Pipeline        | Compose all three: NLI (fast, high precision) → entity gap (surfaces specifics) → SelfCheck (async, for ambiguous cases)          |

**Key insights to keep:**

- **BERTScore and ROUGE-L are blind to hallucination.** A fluent, plausible false claim
  scores as high as a correct answer. These metrics measure similarity, not factual accuracy.
- **SelfCheckGPT works without a reference.** If you have a RAG context, use NLI attribution
  instead (more precise). If you have no context, SelfCheckGPT is the best available tool.
- **Entity-level gaps surface the specific false claim.** Use them to generate the review
  annotation: "Check whether 'Napoleonic Wars' appears in the source material."
- **The threshold is a business decision.** A high-precision threshold means fewer false
  alarms; a high-recall threshold means fewer missed hallucinations. For an editorial
  assistant that sends author-facing content, recall (catching errors) matters more.
- **Hallucination detection does not replace fact-checking.** The guard narrows the
  review queue from 100% of outputs to 5% — but a human must check the flagged ones.

---

### What Riverside Delivers to Authors and Editors

After deploying the hallucination guard, Riverside can commit to:

1. **All HIGH-risk outputs** (NLI attribution < 0.40 or entity gap > 0.50) are held for
   human review before reaching authors.
2. **MEDIUM-risk outputs** are served with a visible confidence indicator: Warning: _"This answer
   references details not fully present in the manuscript catalog — verify before forwarding."_
3. **Quarterly red-team runs** produce a hallucination rate report: the fraction of
   generated answers flagged HIGH or MEDIUM, tracked over model versions.

The Elena Marchetti scenario would have been a MEDIUM-risk output ("Booker Prize 2019 win"
is an entity gap — the award year is in context, but the "won" vs. "longlisted" relation
is not). It would have triggered the Warning: disclaimer before reaching the author.

---

### Complete Hallucination Detection Reference

| Use case                               | Best technique                           | Why                                            | Limitation                                                                |
| -------------------------------------- | ---------------------------------------- | ---------------------------------------------- | ------------------------------------------------------------------------- |
| RAG system with retrieved context      | NLI attribution                          | Uses context as ground truth; no labels needed | Requires NLI model; misses extrinsic hallucination not visible in context |
| Open-ended generation (no context)     | SelfCheckGPT                             | Uses model's own consistency; no reference     | Requires k inference passes; slow at scale                                |
| Surfacing the specific false claim     | Entity gap detection                     | Points to the exact entity to check            | Misses relation-level errors (right entity, wrong relation)               |
| Relation-level errors                  | NLI (contradiction label) + human review | NLI flags the contradiction; human verifies    | Cannot auto-resolve without external knowledge source                     |
| High-recall screening (don't miss any) | Ensemble (NLI + entity + SelfCheck)      | Combines complementary signals                 | Higher false positive rate; expensive                                     |
| Production with latency constraint     | NLI + entity gap only (skip SelfCheck)   | NLI < 100 ms; entity gap < 10 ms               | Misses self-inconsistency detectable only by SelfCheck                    |
